&nbsp;

## [ENCODE-rE2G](https://github.com/EngreitzLab/ENCODE_rE2G) Result Parse Notebook (Heat stress-related datasets)

- [config_biosamples_hs_atacseq_hic_list.tsv](../../out/ENCODE_rE2G/config/config_biosamples_hs_atacseq_hic_list.tsv)で定義した14個のデータセットでの enhancer-gene interaction の相互作用の結果をパースする notebook

In [1]:
import datetime
import os
import pathlib

import polars as pl
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
os.environ["POLARS_MAX_THREADS"] = "16"

In [2]:
execute_date = datetime.datetime.now().strftime("%y%m%d")  # noqa: DTZ005
print(f"Execute Date: {execute_date}")

Execute Date: 261003


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. Parse Target Bed file (Information included: "G"; Gene)

- Target となる領域は [RefSeqCurated.170308.bed.CollapsedGeneBounds.hg38.TSS500bp.bed](https://github.com/EngreitzLab/ENCODE_rE2G/blob/main/reference/RefSeqCurated.170308.bed.CollapsedGeneBounds.hg38.TSS500bp.bed) で定義されている (Download date: 2026-07-31)
- この領域 (TSS 500bp)を通常では遺伝子レベルで表現されているが、これを fanta.bio のBEDファイルとintersect しておくことで、後々の解析をしやすくする
- この領域 (TSS 500bp) と fanta.bio で定義された CRE の重なり具合を算出し、この領域 (TSS 500bp) 側から見たときの割合も算出する (フィルタリング条件に活用するため)
- `ENCODE-rE2G` 側では、範囲が 500 bp と決まっており、この範囲を起点に相互作用が予測されているため、fanta.bio のCREがどの程度カバーしているかを計算しておく
- 同一 `TargetGene` に複数 `cre_id` が紐づく場合は、`tss_cre_overlap_frac_of_tss` が最大の CRE を残す（`pl.when` で明示）
- 同点時は `fantabio_cre_class == "PLA"` を優先し、それでも残る同点は `cre_id` で決定的に1行にする
- 実行コマンド: `cwltool --outdir ./out/ENCODE_rE2G/bedtools_intersect/ ./Tools/bedtools_intersect_v2.cwl ./config/ENCODE_rE2G/encodere2g_refseq_promoter_intersect_fantabio_cre.yml`

In [3]:
# Download https://github.com/EngreitzLab/ENCODE_rE2G/blob/main/reference/RefSeqCurated.170308.bed.CollapsedGeneBounds.hg38.TSS500bp.bed
encodere2g_tatget_refseq_intersect_fantabio_cre = (
    pl.read_csv(
        "../../out/ENCODE_rE2G/bedtools_intersect/RefSeqCurated.170308.bed.CollapsedGeneBounds.hg38.TSS500bp_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed",
        separator="\t",
        has_header=False,
        new_columns=[
            "tss_chr", "tss_start", "tss_end", "TargetGene", "tss_score", "tss_strand",
            "cre_chr", "cre_start", "cre_end", "cre_id", "cre_score", "cre_strand",
            "cre_thick_start", "cre_thick_end", "cre_rgb", "cre_name", "pooled_cre_scores",
            "overlap_bp",  # last column from bedtools -wo
        ]
    ).with_columns(
        pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
        pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
        pl.col("overlap_bp").alias("tss_cre_overlap_bp"),
    ).with_columns(
        (pl.col("tss_cre_overlap_bp") / (pl.col("tss_end") - pl.col("tss_start")))
        .alias("tss_cre_overlap_frac_of_tss"),
    ).with_columns(
        pl.lit("tss_overlap").alias("target_cre_evidence"),
    ).drop("overlap_bp")
)

pl.Config.set_tbl_rows(2)
display(encodere2g_tatget_refseq_intersect_fantabio_cre)

tss_chr,tss_start,tss_end,TargetGene,tss_score,tss_strand,cre_chr,cre_start,cre_end,cre_id,cre_score,cre_strand,cre_thick_start,cre_thick_end,cre_rgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class,tss_cre_overlap_bp,tss_cre_overlap_frac_of_tss,target_cre_evidence
str,i64,i64,str,i64,str,str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str,i64,f64,str
"""chr1""",29120,29620,"""WASH7P""",0,"""-""","""chr1""",29398,29439,"""FCHS_2""",0,""".""",29398,29432,"""255,202,74""","""cp1@WASH7P""","""directionality:1|class:ELA""",1.0,"""ELA""",41,0.082,"""tss_overlap"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",20575460,20575960,"""EIF1AY""",0,"""+""","""chrY""",20575327,20575912,"""FCHS_447281""",0,""".""",20575492,20575772,"""204,0,0""","""cp3@EIF1AY,cp1@TTTY10""","""directionality:0.976236|class:PLA""",0.976236,"""PLA""",452,0.904,"""tss_overlap"""


&nbsp;

&nbsp;

## 1-1: Filter (Information: "G")

- 同一 `TargetGene` に複数 `cre_id` が紐づく場合は、`tss_cre_overlap_frac_of_tss` が最大の CRE を残す（`pl.when` で明示）
- overlap が同点のときは `fantabio_cre_class == "PLA"` を優先する（`pl.when` で `pla_rank` を付与）
- それでも同点のときは `cre_id` の辞書順で決定的に 1 行にする（`sort` + `unique(keep="first")`）
- 分母は ENCODE-rE2G の TSS 500 bp（`tss_end - tss_start`）。閾値による除外は行わず、遺伝子ごとに代表 CRE を 1 本選ぶ

In [4]:
# Keep the CRE with the highest TSS overlap fraction per TargetGene
encodere2g_tatget_refseq_best_cre = (
    encodere2g_tatget_refseq_intersect_fantabio_cre
    .with_columns(
        # Max overlap fraction within each TargetGene
        pl.col("tss_cre_overlap_frac_of_tss")
        .max()
        .over("TargetGene")
        .alias("max_tss_cre_overlap_frac_of_tss"),
    )
    .with_columns(
        # True only for the row(s) that achieve that max
        pl.when(
            pl.col("tss_cre_overlap_frac_of_tss") == pl.col("max_tss_cre_overlap_frac_of_tss")
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("is_best_cre_for_target_gene"),
    )
    .filter(pl.col("is_best_cre_for_target_gene")) # flag

    .with_columns(
        # On ties, prefer PLA (0) over other classes (1)
        pl.when(pl.col("fantabio_cre_class") == "PLA")
        .then(pl.lit(0))
        .otherwise(pl.lit(1))
        .alias("pla_rank"),
    )
    # Tie-break: PLA first, then cre_id for a deterministic single row
    .sort("TargetGene", "pla_rank", "cre_id")
    .unique(subset=["TargetGene"], keep="first")
    .drop("max_tss_cre_overlap_frac_of_tss", "is_best_cre_for_target_gene", "pla_rank")
)

print(
    f"Rows before: {encodere2g_tatget_refseq_intersect_fantabio_cre.height} \n"
    f"Rows after: {encodere2g_tatget_refseq_best_cre.height} \n"
    f"UniqueGenes: {encodere2g_tatget_refseq_best_cre.select('TargetGene').n_unique()}"
)

display(encodere2g_tatget_refseq_best_cre)

Rows before: 38074 
Rows after: 20766 
UniqueGenes: 20766


tss_chr,tss_start,tss_end,TargetGene,tss_score,tss_strand,cre_chr,cre_start,cre_end,cre_id,cre_score,cre_strand,cre_thick_start,cre_thick_end,cre_rgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class,tss_cre_overlap_bp,tss_cre_overlap_frac_of_tss,target_cre_evidence
str,i64,i64,str,i64,str,str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str,i64,f64,str
"""chr19""",58353249,58353749,"""A1BG""",0,"""-""","""chr19""",58353431,58353819,"""FCHS_214135""",0,""".""",58353468,58353803,"""0,0,204""","""cp1@A1BG""","""directionality:-0.999634|class:PLA""",-0.999634,"""PLA""",318,0.636,"""tss_overlap"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr1""",77682408,77682908,"""ZZZ3""",0,"""-""","""chr1""",77682463,77683068,"""FCHS_17233""",0,""".""",77682658,77682722,"""0,0,204""","""cp3@ZZZ3""","""directionality:-0.993424|class:PLA""",-0.993424,"""PLA""",445,0.89,"""tss_overlap"""


&nbsp;

&nbsp;

## Example: LINC00115 / ADAMTS19

In [5]:
# Example: LINC00115 (1行のみ残る)
display(
    encodere2g_tatget_refseq_intersect_fantabio_cre
    .filter(pl.col("TargetGene") == "LINC00115")
    .select("TargetGene", "cre_id", "fantabio_cre_class", "tss_cre_overlap_frac_of_tss")
    .sort("tss_cre_overlap_frac_of_tss", descending=True)
)

display(
    encodere2g_tatget_refseq_best_cre
    .filter(pl.col("TargetGene") == "LINC00115")
    .select("TargetGene", "cre_id", "fantabio_cre_class", "tss_cre_overlap_frac_of_tss")
)


# Example 2: ADAMTS19 (1行のみ残る)
display(
    encodere2g_tatget_refseq_intersect_fantabio_cre
    .filter(pl.col("TargetGene") == "ADAMTS19")
    .select("TargetGene", "cre_id", "fantabio_cre_class", "tss_cre_overlap_frac_of_tss")
    .sort("tss_cre_overlap_frac_of_tss", descending=True)
)

display(
    encodere2g_tatget_refseq_best_cre
    .filter(pl.col("TargetGene") == "ADAMTS19")
    .select("TargetGene", "cre_id", "fantabio_cre_class", "tss_cre_overlap_frac_of_tss")
    .sort("tss_cre_overlap_frac_of_tss", descending=True)
)

TargetGene,cre_id,fantabio_cre_class,tss_cre_overlap_frac_of_tss
str,str,str,f64
"""LINC00115""","""FCHS_26""","""PLA""",0.872
"""LINC00115""","""FCHS_25""","""ELA""",0.278


TargetGene,cre_id,fantabio_cre_class,tss_cre_overlap_frac_of_tss
str,str,str,f64
"""LINC00115""","""FCHS_26""","""PLA""",0.872


TargetGene,cre_id,fantabio_cre_class,tss_cre_overlap_frac_of_tss
str,str,str,f64
"""ADAMTS19""","""FCHS_337546""","""PLA""",1.0
"""ADAMTS19""","""FCHS_337547""","""ELA""",1.0


TargetGene,cre_id,fantabio_cre_class,tss_cre_overlap_frac_of_tss
str,str,str,f64
"""ADAMTS19""","""FCHS_337546""","""PLA""",1.0


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. Integrate `fanta.bio` data (Information included: "E"; enhancer)

- [ENCODE-rE2G](https://github.com/EngreitzLab/ENCODE_rE2G) で得られたエンハンサー候補と、`fanta.bio`のデータを `bedtools intersect` した後の結果をデータフレームとして読み込む
- scatter 版: 複数サンプルの enhancer BED を、共通の fanta.bio CRE peaks と一括 intersect する
- 実行コマンド: `cwltool --outdir ./out/ENCODE_rE2G/bedtools_intersect/enhancer/ ./Workflow/bedtools_intersect_v2_scatter.cwl ./config/ENCODE_rE2G/hs_atacseq_hic_list_re2g_enhancer_intersect_fantabio_cre.yml`
- 単一サンプルのみの場合: `cwltool --outdir ./out/ENCODE_rE2G/bedtools_intersect/enhancer/ ./Tools/bedtools_intersect_v2.cwl ./config/ENCODE_rE2G/gse105028_hs_re2g_enhancer_intersect_fantabio_cre.yml`
- 各行に `re2g_dataset` / `source_bed_file` を付与し、同一 CRE でも由来サンプルを区別できるようにする

In [9]:
# Parse bedtools intersect (-wo) outputs for all enhancer datasets
BED_INTERSECT_ENHANCER_DIR = pathlib.Path("../../out/ENCODE_rE2G/bedtools_intersect/enhancer_hs")

ENHANCER_INTERSECT_COLUMNS = [
    "re2g_chr",
    "re2g_start",
    "re2g_end",
    "cre_chr",
    "cre_start",
    "cre_end",
    "cre_id",
    "cre_score",
    "cre_strand",
    "cre_thick_start",
    "cre_thick_end",
    "cre_rgb",
    "cre_name",
    "pooled_cre_scores",
    "overlap_bp",  # last column from bedtools -wo
]


def read_enhancer_intersect_bed(bed_path: pathlib.Path) -> pl.DataFrame:
    """
    Read one enhancer x fanta.bio intersect BED and tag dataset ID from filename.
    Example stem prefix: GSE101731_HFF_44C_60min
    """
    re2g_dataset = bed_path.stem.split("_encode_e2g_predictions_", 1)[0]

    return (
        pl.read_csv(
            bed_path,
            separator="\t",
            has_header=False,
            new_columns=ENHANCER_INTERSECT_COLUMNS,
            n_threads=16,
        )
        .with_columns(
            pl.col("pooled_cre_scores")
            .str.extract(r"directionality:([^|]+)", 1)
            .cast(pl.Float64, strict=False)
            .alias("directionality"),
            pl.col("pooled_cre_scores")
            .str.extract(r"class:([^|]+)", 1)
            .alias("fantabio_cre_class"),
            pl.col("overlap_bp").alias("enhancer_cre_overlap_bp"),
        )
        .with_columns(
            (pl.col("enhancer_cre_overlap_bp") / (pl.col("cre_end") - pl.col("cre_start"))).alias("enhancer_cre_overlap_frac_of_fantabio_cre"),
            pl.lit(re2g_dataset).alias("re2g_dataset"),
            pl.lit(bed_path.name).alias("source_bed_file"),
        )
        .drop(
            "cre_score",
            "cre_strand",
            "cre_rgb",
            "cre_thick_start",
            "cre_thick_end",
            "pooled_cre_scores",
            "overlap_bp",
        )
    )


bed_paths = sorted(BED_INTERSECT_ENHANCER_DIR.glob("*_intersecting_*.bed"))

enhancer_fanta_intersect = pl.concat(
    [read_enhancer_intersect_bed(p) for p in bed_paths],
    how="vertical_relaxed",
).sort("cre_chr", "cre_start", "re2g_dataset")

print(f"files: {len(bed_paths)}")
print(f"datasets: {enhancer_fanta_intersect.select('re2g_dataset').n_unique()}")

pl.Config.set_tbl_rows(14)
display(
    enhancer_fanta_intersect.group_by("re2g_dataset")
    .len()
    .sort("re2g_dataset")
)
pl.Config.set_tbl_rows(2)
display(enhancer_fanta_intersect)

files: 14
datasets: 14


re2g_dataset,len
str,u32
"""GSE101731_HFF_44C_120min""",94003
"""GSE101731_HFF_44C_60min""",92584
"""GSE105028_hESC_42C_60min""",111014
"""GSE154744_K562_42C_12_5min""",80467
"""GSE154744_K562_42C_25min""",79740
"""GSE154744_K562_42C_30min""",83920
"""GSE154744_K562_42C_30min_intact_hic""",79142
"""GSE154744_K562_42C_40min""",68579
"""GSE154744_K562_42C_60min""",95337


re2g_chr,re2g_start,re2g_end,cre_chr,cre_start,cre_end,cre_id,cre_name,directionality,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,re2g_dataset,source_bed_file
str,i64,i64,str,i64,i64,str,str,f64,str,i64,f64,str,str
"""chr1""",778460,779061,"""chr1""",778536,778928,"""FCHS_19""","""cp5@LINC00115,cp2@LINC01409""",-0.304838,"""PLA""",392,1.0,"""GSE101731_HFF_44C_60min""","""GSE101731_HFF_44C_60min_encode_e2g_predictions_threshold0.179_only_enhancer_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrX""",155612271,155613230,"""chrX""",155613048,155613375,"""FCHS_446859""","""cp1@TMLHE""",-0.947368,"""ELA""",182,0.556575,"""GSE280307_THP1_42C_120min""","""GSE280307_THP1_42C_120min_encode_e2g_predictions_threshold0.179_only_enhancer_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. ENCODE rE2G TSV file (Information Included: "E" / "G")

- このデータは、予測された enhancer - gene interaction が記載されており、`join` する際の 核となるデータセットとして使用される
- `bedpe` 形式とは別の`.tsv.gz` データを読みこんで解析することで、`fanta.bio`との `bedtools intersect`したデータフレームと、それぞれ`join`するための核となるデータとする
- 入力: threshold 適用後の [`../../out/ENCODE_rE2G/hs_atacseq_hic_list/**/encode_e2g_predictions_threshold*.tsv.gz`](../../out/ENCODE_rE2G/hs_atacseq_hic_list)
- 未フィルタ TSV は score 分布が 0 付近に極端に偏るため、プロジェクト既定の threshold 適用後ファイルを用いる
- セクション2の `enhancer_fanta_intersect` と join できるよう、同じ規則で `re2g_dataset` を付与する（例: `GSE101731_HFF_44C_60min`）

In [10]:
# Parse ENCODE-rE2G prediction TSVs for all hs_atacseq_hic_list datasets
RE2G_TSV_ROOT = pathlib.Path("../../out/ENCODE_rE2G/hs_atacseq_hic_list")


def re2g_dataset_from_tsv_path(tsv_path: pathlib.Path) -> str:
    """
    Derive re2g_dataset IDs identical to section 2 enhancer BED tags.
    Examples:
      .../GSE101731_HFF_44C_60min_atacseq/atac_megamap/...
        -> GSE101731_HFF_44C_60min
      .../GSE154744_K562_42C_30min_atacseq_GSE130758_.../atac_intact_hic/...
        -> GSE154744_K562_42C_30min_intact_hic
    """
    biosample_dir = tsv_path.parents[1].name
    mode_dir = tsv_path.parent.name
    if mode_dir == "atac_intact_hic":
        return biosample_dir.split("_atacseq", 1)[0] + "_intact_hic"
    return biosample_dir.split("_atacseq", 1)[0]


def read_re2g_prediction_tsv(tsv_path: pathlib.Path) -> pl.DataFrame:
    """
    Read one thresholded encode_e2g_predictions_threshold*.tsv.gz and tag re2g_dataset.
    Keep normalizedATAC_prom / normalizedH3K27ac_prom as separate columns;
    fill the missing one with null.
    """
    re2g_dataset = re2g_dataset_from_tsv_path(tsv_path)
    df = pl.read_csv(tsv_path, separator="\t", n_threads=16)

    if "normalizedATAC_prom" not in df.columns:
        df = df.with_columns(pl.lit(None).cast(pl.Float64).alias("normalizedATAC_prom"))
    if "normalizedH3K27ac_prom" not in df.columns:
        df = df.with_columns(pl.lit(None).cast(pl.Float64).alias("normalizedH3K27ac_prom"))

    return (
        df.select(
            pl.col("chr").alias("re2g_chr"),
            pl.col("start").alias("re2g_start"),
            pl.col("end").alias("re2g_end"),
            "class",
            "TargetGene",
            "isSelfPromoter",
            "distanceToTSS",
            pl.col("ENCODE-rE2G.Score"),
            pl.struct(
                "TargetGeneTSS",
                "TargetGeneEnsembl_ID",
                "normalizedATAC_prom",
                "normalizedH3K27ac_prom",
                "contactFrequency",
                "numCandidateEnhGene",
                "numTSSEnhGene",
                "sumNearbyEnhancers",
            ).alias("re2g_features"),
            pl.lit(re2g_dataset).alias("re2g_dataset"),
            pl.lit(str(tsv_path)).alias("source_tsv_file"),
        )
        .unique()
    )

In [11]:
# Prefer thresholded predictions; require one file per biosample directory
biosample_dirs = sorted(
    p for p in RE2G_TSV_ROOT.iterdir()
    if p.is_dir() and p.name.startswith("GSE")
)
tsv_paths = sorted(RE2G_TSV_ROOT.glob("**/encode_e2g_predictions_threshold*.tsv.gz"))

missing = []
for biosample_dir in biosample_dirs:
    hits = list(biosample_dir.glob("**/encode_e2g_predictions_threshold*.tsv.gz"))
    if not hits:
        missing.append(biosample_dir.name)

if missing:
    raise FileNotFoundError(
        "Thresholded encode_e2g_predictions_threshold*.tsv.gz missing for: "
        + ", ".join(missing)
    )
if len(tsv_paths) != len(biosample_dirs):
    raise RuntimeError(
        f"Expected {len(biosample_dirs)} thresholded TSVs, found {len(tsv_paths)}"
    )

encode_re2g_predictions = pl.concat(
    [read_re2g_prediction_tsv(p) for p in tsv_paths],
    how="vertical_relaxed",
).sort(["re2g_chr", "re2g_start", "re2g_dataset", "TargetGene"])

pl.Config.set_tbl_rows(14)
print(f"biosamples: {len(biosample_dirs)}")
print(f"files: {len(tsv_paths)}")
print(f"datasets: {encode_re2g_predictions.select('re2g_dataset').n_unique()}")
display(
    encode_re2g_predictions.group_by("re2g_dataset")
    .len()
    .sort("re2g_dataset")
)

# Optional sanity check against section 2 dataset IDs
if "enhancer_fanta_intersect" in globals():
    bed_ids = set(enhancer_fanta_intersect["re2g_dataset"].unique().to_list())
    tsv_ids = set(encode_re2g_predictions["re2g_dataset"].unique().to_list())
    print(f"dataset ID match with section 2: {bed_ids == tsv_ids}")
    if bed_ids != tsv_ids:
        print("only in BED:", sorted(bed_ids - tsv_ids))
        print("only in TSV:", sorted(tsv_ids - bed_ids))

display(encode_re2g_predictions)

biosamples: 14
files: 14
datasets: 14


re2g_dataset,len
str,u32
"""GSE101731_HFF_44C_120min""",75736
"""GSE101731_HFF_44C_60min""",68326
"""GSE105028_hESC_42C_60min""",84031
"""GSE154744_K562_42C_12_5min""",82497
"""GSE154744_K562_42C_25min""",81946
"""GSE154744_K562_42C_30min""",87683
"""GSE154744_K562_42C_30min_intact_hic""",73543
"""GSE154744_K562_42C_40min""",78520
"""GSE154744_K562_42C_60min""",95664


dataset ID match with section 2: True


re2g_chr,re2g_start,re2g_end,class,TargetGene,isSelfPromoter,distanceToTSS,ENCODE-rE2G.Score,re2g_features,re2g_dataset,source_tsv_file
str,i64,i64,str,str,bool,f64,f64,struct[8],str,str
"""chr1""",778160,779409,"""intergenic""","""FAM87B""",false,38585.5,0.686411,"{817370,""ENSG00000177757"",0.309228,null,0.058239,2,2,62.138468}","""GSE173536_lymphblastoid_d21_42C_60min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE173536_lymphblastoid_d21_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
"""chr1""",778160,779409,"""intergenic""","""LINC00115""",false,48737.5,0.484514,"{827522,""ENSG00000225880"",0.55434,null,0.099037,4,4,62.138468}","""GSE173536_lymphblastoid_d21_42C_60min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE173536_lymphblastoid_d21_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
"""chr1""",778160,779409,"""intergenic""","""LINC01128""",false,48805.5,0.48617,"{827590,""ENSG00000228794"",0.550566,null,0.099048,4,4,62.138468}","""GSE173536_lymphblastoid_d21_42C_60min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE173536_lymphblastoid_d21_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
"""chr1""",778163,779660,"""intergenic""","""FAM41C""",false,97890.5,0.187157,"{876802,""ENSG00000230368"",0.32222,null,0.013626,7,5,72.786715}","""GSE173536_lymphblastoid_t21_42C_60min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE173536_lymphblastoid_t21_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
"""chr1""",778163,779660,"""intergenic""","""LINC00115""",false,48610.5,0.458608,"{827522,""ENSG00000225880"",0.532662,null,0.099016,3,4,72.786715}","""GSE173536_lymphblastoid_t21_42C_60min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE173536_lymphblastoid_t21_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
"""chr1""",778163,779660,"""intergenic""","""LINC01128""",false,48678.5,0.458959,"{827590,""ENSG00000228794"",0.531404,null,0.099027,3,4,72.786715}","""GSE173536_lymphblastoid_t21_42C_60min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE173536_lymphblastoid_t21_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
"""chr1""",778168,779084,"""intergenic""","""LINC00115""",false,48896.0,0.305417,"{827522,""ENSG00000225880"",0.475999,null,0.099063,3,4,50.59386}","""GSE209675_MCF7_42C_60min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE209675_MCF7_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
…,…,…,…,…,…,…,…,…,…,…
"""chrX""",155612271,155613230,"""promoter""","""TMLHE""",true,210.5,0.999893,"{155612961,""ENSG00000185973"",0.602729,null,0.232393,0,0,56.027067}","""GSE280307_THP1_42C_120min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE280307_THP1_42C_120min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""


In [12]:
#check
HSPA1A = encode_re2g_predictions.filter(pl.col("TargetGene") == "HSPA1A")
display(HSPA1A)

re2g_chr,re2g_start,re2g_end,class,TargetGene,isSelfPromoter,distanceToTSS,ENCODE-rE2G.Score,re2g_features,re2g_dataset,source_tsv_file
str,i64,i64,str,str,bool,f64,f64,struct[8],str,str
"""chr6""",31812100,31813016,"""genic""","""HSPA1A""",false,2955.0,0.240285,"{31815513,""ENSG00000204389"",0.971693,null,0.168784,1,2,10221.390285}","""GSE173536_lymphblastoid_t21_42C_60min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE173536_lymphblastoid_t21_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
"""chr6""",31813242,31813742,"""genic""","""HSPA1A""",false,2021.0,0.319299,"{31815513,""ENSG00000204389"",0.850697,null,0.168784,1,2,3881.480526}","""GSE154744_K562_42C_40min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE154744_K562_42C_40min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
"""chr6""",31813335,31813835,"""genic""","""HSPA1A""",false,1928.0,0.183925,"{31815513,""ENSG00000204389"",0.887327,null,0.168784,1,2,4436.004236}","""GSE154744_K562_42C_25min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE154744_K562_42C_25min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
"""chr6""",31814090,31815815,"""promoter""","""HSPA1A""",true,560.5,0.999676,"{31815513,""ENSG00000204389"",0.943966,null,0.168784,0,0,4702.115493}","""GSE154744_K562_42C_60min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE154744_K562_42C_60min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
"""chr6""",31814094,31815763,"""promoter""","""HSPA1A""",true,584.5,0.999673,"{31815513,""ENSG00000204389"",0.850697,null,0.168784,0,0,609.916626}","""GSE154744_K562_42C_40min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE154744_K562_42C_40min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
"""chr6""",31814101,31815798,"""promoter""","""HSPA1A""",true,563.5,0.999667,"{31815513,""ENSG00000204389"",0.91411,null,0.168784,0,0,1171.89997}","""GSE154744_K562_42C_12_5min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE154744_K562_42C_12_5min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
"""chr6""",31814112,31815815,"""promoter""","""HSPA1A""",true,549.5,0.99967,"{31815513,""ENSG00000204389"",0.887327,null,0.168784,0,0,382.159906}","""GSE154744_K562_42C_25min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE154744_K562_42C_25min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""
…,…,…,…,…,…,…,…,…,…,…
"""chr6""",31831543,31832656,"""intergenic""","""HSPA1A""",false,16586.5,0.539364,"{31815513,""ENSG00000204389"",0.850794,null,0.061341,5,2,18060.671126}","""GSE101731_HFF_44C_120min""","""../../out/ENCODE_rE2G/hs_atacseq_hic_list/GSE101731_HFF_44C_120min_atacseq/atac_megamap/encode_e2g_predictions_threshold0.179.tsv.gz"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. Join (1) (Information: "E")

- E-G relationships のうち、enhancer 側での統合を以下のセルで実行する

In [13]:
enhancer_fantabio_join = (
    enhancer_fanta_intersect.join(
        encode_re2g_predictions,
        on=["re2g_chr", "re2g_start", "re2g_end", "re2g_dataset"],
        how="inner"
    ).select(
        "cre_id", "cre_name", "cre_chr", "fantabio_cre_class","enhancer_cre_overlap_bp", "enhancer_cre_overlap_frac_of_fantabio_cre","TargetGene", "class","isSelfPromoter", "distanceToTSS", "ENCODE-rE2G.Score", "re2g_dataset",
        pl.struct("re2g_chr", "re2g_start", "re2g_end").alias("re2g_enhancer_locus"), # まとめる
        pl.struct("cre_start", "cre_end", "directionality").alias("fantabio_enhancer_cre"), # まとめる
        "re2g_features"  # already a struct from encode side
    )
).sort(["cre_id", "cre_name", "TargetGene", "re2g_dataset"])

display(enhancer_fantabio_join)

cre_id,cre_name,cre_chr,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,TargetGene,class,isSelfPromoter,distanceToTSS,ENCODE-rE2G.Score,re2g_dataset,re2g_enhancer_locus,fantabio_enhancer_cre,re2g_features
str,str,str,str,i64,f64,str,str,bool,f64,f64,str,struct[3],struct[3],struct[8]
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""promoter""",true,457.5,0.999461,"""GSE105028_hESC_42C_60min""","{""chr1"",960336,961751}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",null,0.576527,0.236217,0,0,170.367064}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",74,0.155462,"""KLHL17""","""promoter""",true,76.5,0.999967,"""GSE154744_K562_42C_25min""","{""chr1"",960302,961023}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.512267,null,0.259602,0,0,244.856436}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",76,0.159664,"""KLHL17""","""promoter""",true,84.0,0.999959,"""GSE154744_K562_42C_40min""","{""chr1"",960315,961025}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.572631,null,0.259602,0,0,282.531976}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",118,0.247899,"""KLHL17""","""promoter""",true,108.5,0.999955,"""GSE154744_K562_42C_60min""","{""chr1"",960322,961067}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.503581,null,0.259602,0,0,313.411192}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""promoter""",true,364.0,0.999857,"""GSE173536_lymphblastoid_d21_42C_60min""","{""chr1"",960335,961565}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.478225,null,0.259602,0,0,209.961424}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",293,0.615546,"""KLHL17""","""promoter""",true,196.5,0.999926,"""GSE173536_lymphblastoid_t21_42C_60min""","{""chr1"",960323,961242}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.461337,null,0.259602,0,0,233.348032}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",87,0.182773,"""KLHL17""","""promoter""",true,94.5,0.999971,"""GSE209675_MCF7_42C_60min""","{""chr1"",960325,961036}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.377504,null,0.259602,0,0,128.99544}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_99994""","""cp1@LRRIQ1""","""chr12""","""ELA""",96,1.0,"""TSPAN19""","""promoter""",true,318.5,0.999844,"""GSE101731_HFF_44C_60min""","{""chr12"",85036027,85037164}","{85036596,85036692,0.333333}","{85036277,""ENSG00000231738"",0.464773,null,0.192534,0,0,0.0}"


In [14]:
print(f"ENCODE-rE2G overlap 1st quartile: {enhancer_fantabio_join.select("enhancer_cre_overlap_frac_of_fantabio_cre").to_series().quantile(0.25)}")
print(f"ENCODE-rE2G overlap median: {enhancer_fantabio_join.select("enhancer_cre_overlap_frac_of_fantabio_cre").to_series().median()}")
print(f"ENCODE-rE2G overlap 3rd quartile: {enhancer_fantabio_join.select("enhancer_cre_overlap_frac_of_fantabio_cre").to_series().quantile(0.75)}")

print(f"ENCODE-rE2G overlap mean: {enhancer_fantabio_join.select("ENCODE-rE2G.Score").to_series().mean()}")
print(f"ENCODE-rE2G overlap standard deviation: {enhancer_fantabio_join.select("ENCODE-rE2G.Score").to_series().std()}")

ENCODE-rE2G overlap 1st quartile: 0.6130790190735694
ENCODE-rE2G overlap median: 0.9851485148514851
ENCODE-rE2G overlap 3rd quartile: 1.0
ENCODE-rE2G overlap mean: 0.6655196171701648
ENCODE-rE2G overlap standard deviation: 0.32213403720084915


&nbsp;

&nbsp;

### ENCODE rE2G score

- 中央値、第3四分位数などを調べておき、フィルタリングの条件に使用する
- ただし、それぞれのデータセットでは、異なる機械学習モデルを使用しており、あくまで参考程度に留めるべき値だと考えられる

In [15]:
print(f"ENCODE-rE2G score 1st quartile: {enhancer_fantabio_join['ENCODE-rE2G.Score'].quantile(0.25)}")
print(f"ENCODE-rE2G score median: {enhancer_fantabio_join['ENCODE-rE2G.Score'].median()}")
print(f"ENCODE-rE2G score 3rd quartile: {enhancer_fantabio_join['ENCODE-rE2G.Score'].quantile(0.75)}")
print(f"ENCODE-rE2G score mean: {enhancer_fantabio_join['ENCODE-rE2G.Score'].mean()}")
print(f"ENCODE-rE2G score standard deviation: {enhancer_fantabio_join['ENCODE-rE2G.Score'].std()}")

ENCODE-rE2G score 1st quartile: 0.3254861794176
ENCODE-rE2G score median: 0.7179793744815547
ENCODE-rE2G score 3rd quartile: 0.999678266372826
ENCODE-rE2G score mean: 0.6655196171701648
ENCODE-rE2G score standard deviation: 0.32213403720084915


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 5. Filter (1) (Information: "E") (IMPORTANT!)

- `ENCODE-rE2G`で予測された enhancer 領域と、`fanta.bio`で定義されているCREのカバー率に基づいてフィルタリング
- fanta.bio の CRE とのカバー率でフィルタリング
- `enhancer_cre_overlap_bp` / (`cre_end` - `cre_start`)

In [16]:
enhancer_fantabio_join_filter = enhancer_fantabio_join.filter(
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre") >= 0.5
)

display(enhancer_fantabio_join_filter)

cre_id,cre_name,cre_chr,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,TargetGene,class,isSelfPromoter,distanceToTSS,ENCODE-rE2G.Score,re2g_dataset,re2g_enhancer_locus,fantabio_enhancer_cre,re2g_features
str,str,str,str,i64,f64,str,str,bool,f64,f64,str,struct[3],struct[3],struct[8]
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""promoter""",true,457.5,0.999461,"""GSE105028_hESC_42C_60min""","{""chr1"",960336,961751}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",null,0.576527,0.236217,0,0,170.367064}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""promoter""",true,364.0,0.999857,"""GSE173536_lymphblastoid_d21_42C_60min""","{""chr1"",960335,961565}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.478225,null,0.259602,0,0,209.961424}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",293,0.615546,"""KLHL17""","""promoter""",true,196.5,0.999926,"""GSE173536_lymphblastoid_t21_42C_60min""","{""chr1"",960323,961242}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.461337,null,0.259602,0,0,233.348032}"
"""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""promoter""",true,371.0,0.999851,"""GSE267931_HUVEC_39C_120min""","{""chr1"",960336,961578}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.501016,null,0.259602,0,0,415.927582}"
"""FCHS_1000""","""cp2@RER1""","""chr1""","""ELA""",378,0.574468,"""MORN1""","""promoter""",true,701.0,0.999568,"""GSE154744_K562_42C_25min""","{""chr1"",2392202,2392702}","{2392324,2392982,0.0}","{2391751,""ENSG00000116151"",0.859092,null,0.207712,0,0,1270.747782}"
"""FCHS_1000""","""cp2@RER1""","""chr1""","""ELA""",378,0.574468,"""RER1""","""promoter""",true,678.0,0.998301,"""GSE154744_K562_42C_25min""","{""chr1"",2392202,2392702}","{2392324,2392982,0.0}","{2391774,""ENSG00000157916"",0.859842,null,0.207712,0,0,1270.747782}"
"""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",353,0.842482,"""AZIN2""","""genic""",false,15223.0,0.342709,"""GSE154744_K562_42C_25min""","{""chr1"",33097150,33097650}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.383964,null,0.044833,2,1,220.597881}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_99994""","""cp1@LRRIQ1""","""chr12""","""ELA""",96,1.0,"""LRRIQ1""","""promoter""",true,93.0,0.99996,"""GSE173536_lymphblastoid_t21_42C_60min""","{""chr12"",85036027,85036799}","{85036596,85036692,0.333333}","{85036320,""ENSG00000133640"",0.409078,null,0.192534,0,0,0.0}"


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 6. Join (2) (Information: "G")

- target 側のデータ (fanta.bio との統合をおこなった状態)を統合する

&nbsp;

## 6-1: Fomatting for `join`

In [17]:
encodere2g_tatget_refseq_best_cre_select = encodere2g_tatget_refseq_best_cre.select(
    "TargetGene",
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    pl.struct("tss_chr", "tss_start", "tss_end", "tss_score", "tss_strand").alias("target_tss_info"),
    pl.struct("cre_chr", "cre_start", "cre_end", "cre_score", "cre_strand", "cre_thick_start", "cre_thick_end", "cre_rgb", "pooled_cre_scores", "directionality").alias("target_cre_info"),
    pl.struct("tss_cre_overlap_bp", "tss_cre_overlap_frac_of_tss").alias("target_cre_overlap_info")
).rename(
    {
        "cre_id": "target_cre_id",
        "cre_name": "target_cre_name",
        "fantabio_cre_class": "target_fantabio_cre_class"      
    }
)

display(encodere2g_tatget_refseq_best_cre_select)

TargetGene,target_cre_id,target_cre_name,target_fantabio_cre_class,target_tss_info,target_cre_info,target_cre_overlap_info
str,str,str,str,struct[5],struct[10],struct[2]
"""A1BG""","""FCHS_214135""","""cp1@A1BG""","""PLA""","{""chr19"",58353249,58353749,0,""-""}","{""chr19"",58353431,58353819,0,""."",58353468,58353803,""0,0,204"",""directionality:-0.999634|class:PLA"",-0.999634}","{318,0.636}"
"""A1CF""","""FCHS_49959""","""cp1@A1CF""","""PLA""","{""chr10"",50885425,50885925,0,""-""}","{""chr10"",50885528,50885774,0,""."",50885634,50885770,""0,0,204"",""directionality:-0.999645|class:PLA"",-0.999645}","{246,0.492}"
"""A2M""","""FCHS_88190""","""cp1@A2M""","""PLA""","{""chr12"",9115979,9116479,0,""-""}","{""chr12"",9115751,9116452,0,""."",9115913,9116283,""0,0,204"",""directionality:-0.999764|class:PLA"",-0.999764}","{473,0.946}"
"""A2M-AS1""","""FCHS_88157""","""cp2@A2M-AS1,cp1@LINC00612""","""PLA""","{""chr12"",9064926,9065426,0,""+""}","{""chr12"",9064873,9065385,0,""."",9065079,9065268,""0,0,102"",""directionality:-0.480891|class:PLA"",-0.480891}","{459,0.918}"
"""A2ML1""","""FCHS_88093""","""cp7@A2ML1""","""PLA""","{""chr12"",8822221,8822721,0,""+""}","{""chr12"",8822444,8822725,0,""."",8822444,8822620,""255,0,0"",""directionality:1|class:PLA"",1.0}","{277,0.554}"
"""A4GALT""","""FCHS_276044""","""cp4@A4GALT,cp5@CYB5R3""","""PLA""","{""chr22"",42720660,42721160,0,""-""}","{""chr22"",42720635,42721121,0,""."",42720828,42720942,""0,0,204"",""directionality:-0.999079|class:PLA"",-0.999079}","{461,0.922}"
"""AAAS""","""FCHS_95187""","""cp11@AAAS""","""PLA""","{""chr12"",53321378,53321878,0,""-""}","{""chr12"",53321460,53322000,0,""."",53321591,53321702,""0,0,204"",""directionality:-0.989867|class:PLA"",-0.989867}","{418,0.836}"
…,…,…,…,…,…,…
"""ZXDB""","""FCHS_440127""","""cp3@ZXDB""","""ELA""","{""chrX"",57591585,57592085,0,""+""}","{""chrX"",57591492,57591853,0,""."",57591614,57591733,""255,202,74"",""directionality:0.939394|class:ELA"",0.939394}","{268,0.536}"


&nbsp;

&nbsp;

## 6-2: Join

In [18]:
e2g_join = enhancer_fantabio_join_filter.join(
    encodere2g_tatget_refseq_best_cre_select,
    on=["TargetGene"],
    how="inner"
).select(
    # 見やすいように列順を変更
    "ENCODE-rE2G.Score",
    "re2g_dataset",
    "cre_id",
    "cre_name",
    "cre_chr",
    "fantabio_cre_class",
    "enhancer_cre_overlap_bp",
    "enhancer_cre_overlap_frac_of_fantabio_cre",
    "TargetGene",
    "target_cre_id",
    "target_cre_name",
    "target_fantabio_cre_class",
    "class",
    "isSelfPromoter",
    "distanceToTSS",
    "re2g_enhancer_locus",
    "fantabio_enhancer_cre",
    "re2g_features",
    "target_tss_info",
    "target_cre_info",
    "target_cre_overlap_info"
)

e2g_join.write_parquet(
    # 保存するのは、フィルタリング後のデータ
    f"../../out/ENCODE_rE2G/parquet_file/encode_re2g_hs_join_v{execute_date}.parquet"
)

display(e2g_join)

ENCODE-rE2G.Score,re2g_dataset,cre_id,cre_name,cre_chr,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,TargetGene,target_cre_id,target_cre_name,target_fantabio_cre_class,class,isSelfPromoter,distanceToTSS,re2g_enhancer_locus,fantabio_enhancer_cre,re2g_features,target_tss_info,target_cre_info,target_cre_overlap_info
f64,str,str,str,str,str,i64,f64,str,str,str,str,str,bool,f64,struct[3],struct[3],struct[8],struct[5],struct[10],struct[2]
0.999461,"""GSE105028_hESC_42C_60min""","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,457.5,"{""chr1"",960336,961751}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",null,0.576527,0.236217,0,0,170.367064}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999857,"""GSE173536_lymphblastoid_d21_42C_60min""","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,364.0,"{""chr1"",960335,961565}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.478225,null,0.259602,0,0,209.961424}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999926,"""GSE173536_lymphblastoid_t21_42C_60min""","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",293,0.615546,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,196.5,"{""chr1"",960323,961242}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.461337,null,0.259602,0,0,233.348032}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999851,"""GSE267931_HUVEC_39C_120min""","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,371.0,"{""chr1"",960336,961578}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.501016,null,0.259602,0,0,415.927582}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999568,"""GSE154744_K562_42C_25min""","""FCHS_1000""","""cp2@RER1""","""chr1""","""ELA""",378,0.574468,"""MORN1""","""FCHS_999""","""cp1@MORN1,cp1@RER1""","""PLA""","""promoter""",true,701.0,"{""chr1"",2392202,2392702}","{2392324,2392982,0.0}","{2391751,""ENSG00000116151"",0.859092,null,0.207712,0,0,1270.747782}","{""chr1"",2391501,2392001,0,""-""}","{""chr1"",2391399,2392038,0,""."",2391554,2391837,""0,0,0"",""directionality:-0.131564|class:PLA"",-0.131564}","{500,1.0}"
0.998301,"""GSE154744_K562_42C_25min""","""FCHS_1000""","""cp2@RER1""","""chr1""","""ELA""",378,0.574468,"""RER1""","""FCHS_999""","""cp1@MORN1,cp1@RER1""","""PLA""","""promoter""",true,678.0,"{""chr1"",2392202,2392702}","{2392324,2392982,0.0}","{2391774,""ENSG00000157916"",0.859842,null,0.207712,0,0,1270.747782}","{""chr1"",2391524,2392024,0,""+""}","{""chr1"",2391399,2392038,0,""."",2391554,2391837,""0,0,0"",""directionality:-0.131564|class:PLA"",-0.131564}","{500,1.0}"
0.342709,"""GSE154744_K562_42C_25min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",353,0.842482,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15223.0,"{""chr1"",33097150,33097650}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.383964,null,0.044833,2,1,220.597881}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.99996,"""GSE173536_lymphblastoid_t21_42C_60min""","""FCHS_99994""","""cp1@LRRIQ1""","""chr12""","""ELA""",96,1.0,"""LRRIQ1""",""

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 7.  Enhancer-target Region Unique Count

- enhancer 側で統合した `fanta.bio` の `cre_id` と Target 側の領域として統合した `fanta.bio`の `target_cre_id` の unique pair をカウントする
- `re2g_dataset` カラムのカウントも行う

In [19]:
e2g_join_unique_count = e2g_join.group_by(
    "cre_id",
    "fantabio_cre_class",
    "target_cre_id",
    "target_fantabio_cre_class",
    # "isSelfPromoter" (この値は遺伝子ごとに異なるのでグループ化しない)
).agg(
    pl.col("re2g_dataset").unique().alias("re2g_dataset"),
    pl.col("re2g_dataset").n_unique().alias("re2g_dataset_count"),
    pl.col("ENCODE-rE2G.Score").alias("ENCODE-rE2G.Score"),
    
    pl.col("ENCODE-rE2G.Score").min().alias("ENCODE-rE2G.Score_min"),
    pl.col("ENCODE-rE2G.Score").max().alias("ENCODE-rE2G.Score_max"),
    pl.col("ENCODE-rE2G.Score").mean().alias("ENCODE-rE2G.Score_mean"),

    # pl.col("ENCODE-rE2G.Score").quantile(0.25).alias("ENCODE-rE2G.Score_q1"),
    # pl.col("ENCODE-rE2G.Score").median().alias("ENCODE-rE2G.Score_median"),
    # pl.col("ENCODE-rE2G.Score").quantile(0.75).alias("ENCODE-rE2G.Score_q3"),
    # pl.col("ENCODE-rE2G.Score").std().alias("ENCODE-rE2G.Score_std"),
    
).sort(
    by=["re2g_dataset_count"],
    descending=True
)


print(f"ENCODE-rE2G enhancer - target region interaction unique pair count: {e2g_join_unique_count.select(['cre_id', 'target_cre_id']).n_unique()}")
display(e2g_join_unique_count)

ENCODE-rE2G enhancer - target region interaction unique pair count: 423357


cre_id,fantabio_cre_class,target_cre_id,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,list[str],u32,list[f64],f64,f64,f64
"""FCHS_355223""","""PLA""","""FCHS_355223""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.999932, 0.999914, … 0.999907]",0.99825,1.0,0.999756
"""FCHS_459587""","""ELA""","""FCHS_39219""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.999839, 0.999454, … 0.999725]",0.985544,0.999864,0.998699
"""FCHS_373166""","""PLA""","""FCHS_373166""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.999796, 0.999728, … 0.999864]",0.996349,0.999864,0.999531
"""FCHS_231027""","""ELA""","""FCHS_231026""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.998943, 0.999308, … 0.999576]",0.991033,0.999879,0.998553
"""FCHS_174697""","""PLA""","""FCHS_174697""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.999951, 0.999967, … 0.999962]",0.997844,0.999973,0.999627
"""FCHS_189431""","""PLA""","""FCHS_189433""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.999919, 0.999912, … 0.99986]",0.998462,0.99997,0.999701
"""FCHS_257469""","""PLA""","""FCHS_257470""","""ELA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.999758, 0.999452, … 0.99945]",0.997888,0.999844,0.99958
…,…,…,…,…,…,…,…,…,…
"""FCHS_585135""","""ELA""","""FCHS_420037""","""PLA""","[""GSE280307_THP1_42C_120min""]",1,[0.213788],0.213788,0.213788,0.213788


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 8. Check `isSelfPromoter`

&nbsp;

&nbsp;

&nbsp;

## 8-0: Selection of proximal enhancer

- `isSelfPromoter == True` でフィルタリングする

In [32]:
e2g_join_self_promoter = e2g_join.filter(
    pl.col("isSelfPromoter") == True
)

e2g_join_self_promoter.write_parquet(
    f"../../out/ENCODE_rE2G/parquet_file/encode_re2g_hs_join_self_promoter_v{execute_date}.parquet"
)

display(e2g_join_self_promoter)

ENCODE-rE2G.Score,re2g_dataset,cre_id,cre_name,cre_chr,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,TargetGene,target_cre_id,target_cre_name,target_fantabio_cre_class,class,isSelfPromoter,distanceToTSS,re2g_enhancer_locus,fantabio_enhancer_cre,re2g_features,target_tss_info,target_cre_info,target_cre_overlap_info
f64,str,str,str,str,str,i64,f64,str,str,str,str,str,bool,f64,struct[3],struct[3],struct[8],struct[5],struct[10],struct[2]
0.999461,"""GSE105028_hESC_42C_60min""","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,457.5,"{""chr1"",960336,961751}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",null,0.576527,0.236217,0,0,170.367064}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999857,"""GSE173536_lymphblastoid_d21_42C_60min""","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,364.0,"{""chr1"",960335,961565}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.478225,null,0.259602,0,0,209.961424}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999926,"""GSE173536_lymphblastoid_t21_42C_60min""","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",293,0.615546,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,196.5,"{""chr1"",960323,961242}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.461337,null,0.259602,0,0,233.348032}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999851,"""GSE267931_HUVEC_39C_120min""","""FCHS_100""","""cp6@KLHL17""","""chr1""","""ELA""",476,1.0,"""KLHL17""","""FCHS_98""","""cp3@KLHL17,cp7@NOC2L""","""PLA""","""promoter""",true,371.0,"{""chr1"",960336,961578}","{960949,961425,-0.483871}","{960586,""ENSG00000187961"",0.501016,null,0.259602,0,0,415.927582}","{""chr1"",960336,960836,0,""+""}","{""chr1"",960174,960791,0,""."",960534,960591,""204,0,0"",""directionality:0.864177|class:PLA"",0.864177}","{455,0.91}"
0.999568,"""GSE154744_K562_42C_25min""","""FCHS_1000""","""cp2@RER1""","""chr1""","""ELA""",378,0.574468,"""MORN1""","""FCHS_999""","""cp1@MORN1,cp1@RER1""","""PLA""","""promoter""",true,701.0,"{""chr1"",2392202,2392702}","{2392324,2392982,0.0}","{2391751,""ENSG00000116151"",0.859092,null,0.207712,0,0,1270.747782}","{""chr1"",2391501,2392001,0,""-""}","{""chr1"",2391399,2392038,0,""."",2391554,2391837,""0,0,0"",""directionality:-0.131564|class:PLA"",-0.131564}","{500,1.0}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.999937,"""GSE101731_HFF_44C_120min""","""FCHS_99994""","""cp1@LRRIQ1""","""chr12""","""ELA""",96,1.0,"""TSPAN19""","""FCHS_99993""","""cp4@LRRIQ1,cp3@TSPAN19""","""PLA""","""promoter""",true,146.0,"{""chr12"",85036027,85036819}","{85036596,85036692,0.333333}","{85036277,""ENSG00000231738"",0.410312,null,0.192534,0,0,0.0}","{""chr12"",85036027,85036527,0,""-""}","{""chr12"",85036236,85036498,0,""."",85036260,85036344,""204,0,0"",""directionality:0.970459|class:PLA"",0.970459}","{262,0.524}"
0.999844,"""GSE101731_HFF_44C_60min""","""FCHS_99994""","""cp1@LRRIQ1""","""chr12""","""ELA""",96,1.0,"""TSPAN19""","""FCHS_99993""","""cp4@LRRIQ1,cp3@TSPAN19""","""PLA""","""promoter""",true,318.5,"{""chr12"",85036027,85037164}","{85036596,85036692,0.333333}","{85036277,""ENSG00000231738"",0.464773,null,0.192534,0,0,0.0}","{""chr12"",85036027,85036527,0,""-""}","{""chr12"",85036236,85036498,0,""."",85036260,85036344,""204,0,0"",""directionality:0.970459|class:PLA"",0.970459}","{262,0.524}"
0.999993,"""GSE173536_lymphblastoid_d21_42C_60min""","""FCHS_99994""","""cp1@LRRIQ1""","""chr1

&nbsp;

&nbsp;

&nbsp;

## 8-1: Selection of distal enhancer (Important!)

- `TargetGene` の近傍ではない enhancer を標的とする
- `isSelfPromoter` == False でフィルタリング

In [31]:
e2g_join_filter = e2g_join.filter(
    pl.col("isSelfPromoter") == False
)

e2g_join_filter.write_parquet(
    f"../../out/ENCODE_rE2G/parquet_file/encode_re2g_hs_join_filter_distal_enhancer_v{execute_date}.parquet"
)

display(e2g_join_filter)

ENCODE-rE2G.Score,re2g_dataset,cre_id,cre_name,cre_chr,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,TargetGene,target_cre_id,target_cre_name,target_fantabio_cre_class,class,isSelfPromoter,distanceToTSS,re2g_enhancer_locus,fantabio_enhancer_cre,re2g_features,target_tss_info,target_cre_info,target_cre_overlap_info
f64,str,str,str,str,str,i64,f64,str,str,str,str,str,bool,f64,struct[3],struct[3],struct[8],struct[5],struct[10],struct[2]
0.342709,"""GSE154744_K562_42C_25min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",353,0.842482,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15223.0,"{""chr1"",33097150,33097650}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.383964,null,0.044833,2,1,220.597881}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
0.320048,"""GSE154744_K562_42C_60min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",288,0.687351,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15288.0,"{""chr1"",33097215,33097715}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.402521,null,0.044762,4,1,253.0904}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
0.312414,"""GSE267931_HUVEC_39C_120min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",358,0.854415,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15218.0,"{""chr1"",33097145,33097645}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.466515,null,0.044839,4,1,506.041996}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
0.194134,"""GSE267931_HUVEC_39C_120min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",358,0.854415,"""TRIM62""","""FCHS_10021""","""cp3@TRIM62""","""PLA""","""genic""",false,84675.0,"{""chr1"",33097145,33097645}","{33097084,33097503,0.614679}","{33182070,""ENSG00000116525"",0.524194,null,0.013442,13,1,506.041996}","{""chr1"",33181820,33182320,0,""-""}","{""chr1"",33181859,33182582,0,""."",33182047,33182188,""0,0,204"",""directionality:-0.999247|class:PLA"",-0.999247}","{461,0.922}"
0.291194,"""GSE173536_lymphblastoid_d21_42C_60min""","""FCHS_100026""","""FCHS_100026""","""chr12""","""ELA""",201,1.0,"""RASSF9""","""FCHS_100044""","""cp1@RASSF9""","""PLA""","""intergenic""",false,411922.0,"{""chr12"",85423911,85425325}","{85423957,85424158,0.5}","{85836540,""ENSG00000198774"",0.357834,null,0.00446,9,1,0.0}","{""chr12"",85836290,85836790,0,""-""}","{""chr12"",85836188,85836732,0,""."",85836213,85836732,""0,0,255"",""directionality:-1|class:PLA"",-1.0}","{442,0.884}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.644669,"""GSE101731_HFF_44C_60min""","""FCHS_99987""","""cp1@TSPAN19""","""chr12""","""PLA""",423,1.0,"""LRRIQ1""","""FCHS_99993""","""cp4@LRRIQ1,cp3@TSPAN19""","""PLA""","""genic""",false,19724.0,"{""chr12"",85016346,85016846}","{85016374,85016797,-0.977465}","{85036320,""ENSG00000133640"",0.470943,null,0.036875,2,2,0.0}","{""chr12"",85036070,85036570,0,""+""}","{""chr12"",85036236,85036498,0,""."",85036260,85036344,""204,0,0"",""directionality:0.970459|class:PLA"",0.970459}","{262,0.524}"
0.311435,"""GSE101731_HFF_44C_120min""","""FCHS_99987""","""cp1@TSPAN19""","""chr12""","""PLA""",423,1.0,"""SLC6A15""","""FCHS_99974""","""cp9@SLC6A15""","""PLA""","""genic""",false,103772.0,"{""chr12"",85016351,85016851}","{85016374,85016797,-0.977465}","{84912829,""ENSG00000072041"",0.486403,null,0.014404,3,1,0.0}","{""chr12"",84912579,84913079,0,""-""}","{""chr12"",84912565,84913337,0,""."",84912757,84912903,""0,0,204"",""directionality:-0.99533|class:PLA"",-0.99533}","{500,1.0}"
0.291244,"""GSE101731_HFF_44C_60min""","""FCHS_99987""","""cp1@TSPA

In [30]:
e2g_join_filter_unique_count = e2g_join_filter.group_by(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "target_cre_id",
    "target_cre_name",
    "target_fantabio_cre_class"
).agg(
    pl.col("re2g_dataset").unique().alias("re2g_dataset"),
    pl.col("re2g_dataset").n_unique().alias("re2g_dataset_count"),
    pl.col("ENCODE-rE2G.Score").alias("ENCODE-rE2G.Score"),
    
    pl.col("ENCODE-rE2G.Score").min().alias("ENCODE-rE2G.Score_min"),
    pl.col("ENCODE-rE2G.Score").max().alias("ENCODE-rE2G.Score_max"),
    pl.col("ENCODE-rE2G.Score").mean().alias("ENCODE-rE2G.Score_mean"),

    # pl.col("ENCODE-rE2G.Score").quantile(0.25).alias("ENCODE-rE2G.Score_q1"),
    # pl.col("ENCODE-rE2G.Score").median().alias("ENCODE-rE2G.Score_median"),
    # pl.col("ENCODE-rE2G.Score").quantile(0.75).alias("ENCODE-rE2G.Score_q3"),
    # pl.col("ENCODE-rE2G.Score").std().alias("ENCODE-rE2G.Score_std"),
).sort(
    by=["re2g_dataset_count"],
    descending=True
)

pl.Config.set_tbl_rows(10)
print(f"ENCODE-rE2G enhancer - target region interaction unique pair count: {e2g_join_filter_unique_count.select(['cre_id', 'target_cre_id']).n_unique()}")
display(e2g_join_filter_unique_count)

ENCODE-rE2G enhancer - target region interaction unique pair count: 368398


cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64
"""FCHS_235807""","""cp13@NR4A2""","""PLA""","""FCHS_235801""","""cp3@NR4A2""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.930365, 0.953361, … 0.957654]",0.872632,0.971833,0.949118
"""FCHS_269918""","""cp2@LOC107985532""","""PLA""","""FCHS_269908""","""cp3@SDF2L1""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.394801, 0.601796, … 0.364564]",0.317749,0.817551,0.518851
"""FCHS_58710""","""FCHS_58710""","""ELA""","""FCHS_58719""","""cp2@STN1""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.634674, 0.505039, … 0.672167]",0.415113,0.908999,0.62206
"""FCHS_280150""","""cp5@SLC6A6""","""PLA""","""FCHS_280126""","""cp4@SLC6A6""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.658301, 0.760305, … 0.908059]",0.29959,0.979908,0.688886
"""FCHS_414059""","""cp3@ASTILCS,cp14@PTP4A3""","""PLA""","""FCHS_414048""","""cp1@GPR20""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.481073, 0.701595, … 0.848789]",0.481073,0.890794,0.755901
…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_147780""","""cp2@HBM""","""ELA""","""FCHS_147791""","""cp1@HBQ1""","""PLA""","[""GSE280307_THP1_42C_120min""]",1,[0.180577],0.180577,0.180577,0.180577
"""FCHS_418892""","""FCHS_418892""","""ELA""","""FCHS_418924""","""cp7@DNAI1,cp1@FAM219A""","""PLA""","[""GSE105028_hESC_42C_60min""]",1,[0.281531],0.281531,0.281531,0.281531
"""FCHS_514577""","""FCHS_514577""","""ELA""","""FCHS_206213""","""cp3@YIF1B""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,[0.2057],0.2057,0.2057,0.2057


&nbsp;

&nbsp;

&nbsp;

## 8-2: Join

In [ ]:
# fantabio_HN2_up5p_diff_analysis_atac = pl.read_parquet(
#     "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/fantabio_HN2_up5p_v260721_diff_analysis_atac.parquet"
# )

# pl.Config.set_tbl_rows(2)
# display(fantabio_HN2_up5p_diff_analysis_atac)

In [ ]:
# # Keep fantabio rows whose cre_id appears in e2g_join
# e2g_cre_ids = e2g_join_filter_unique_count.select("cre_id").unique()


# # Use implode() so is_in receives a List (avoids DeprecationWarning in Polars 1.40+)
# fantabio_HN2_up5p_in_e2g = fantabio_HN2_up5p_diff_analysis_atac.filter(
#     pl.col("cre_id").is_in(pl.lit(e2g_cre_ids.get_column("cre_id")).implode())
# )

# print(
#     f"up5p transcript - CRE pairs: {fantabio_HN2_up5p_diff_analysis_atac.select(["cre_id", "transcript_id", "HN2"]).unique().n_unique()} \n"
#     f"e2g unique cre_id (after filtering): {e2g_cre_ids.select(["cre_id"]).n_unique()} \n"
#     f"Filtered up5p transcript - CRE pairs: {fantabio_HN2_up5p_in_e2g.select(["cre_id", "transcript_id", "HN2"]).unique().n_unique()}"
# )

# pl.Config.set_tbl_rows(2)
# display(fantabio_HN2_up5p_in_e2g)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 8-3: Enhancer-derived lncRNA

- `fantabio_HN2_up5p_in_e2g` データフレームから、[../integrated_analysis/enhancer_derived_lncRNA.ipynb](../integrated_analysis/enhancer_derived_lncRNA.ipynb) で作成した enhancer-derived lncRNA のエントリーのみを選択する

In [ ]:
# enhancer_derived_lncRNA = pl.read_csv(
#     "../../out/integrated_analysis/enhancer_derived_lncRNA/enhancer_derived_lncrna_transcript_list.tsv",
#     separator="\t",
#     has_header=False,
#     new_columns=["cre_id", "transcript_id"]
# )

# print(f"unique lncRNA transcript_id: {enhancer_derived_lncRNA.select('transcript_id').n_unique()}")
# display(enhancer_derived_lncRNA)

In [ ]:
# # only lncRNA
# fantabio_HN2_up5p_in_e2g_lncRNA = fantabio_HN2_up5p_in_e2g.filter(
#     (pl.col("transcript_type") == "lncRNA") &
#     (pl.col("fantabio_cre_class") == "ELA")
# ).filter(
#     pl.col("transcript_id").is_in(enhancer_derived_lncRNA.get_column("transcript_id").implode())
# )


# pl.Config.set_tbl_rows(6)
# print(f"unique lncRNA cre_id: {fantabio_HN2_up5p_in_e2g_lncRNA.select('cre_id').n_unique()}")
# print(f"unique lncRNA transcript_id: {fantabio_HN2_up5p_in_e2g_lncRNA.select('transcript_id').n_unique()}")
# display(fantabio_HN2_up5p_in_e2g_lncRNA)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 8-4: JOIN (lncRNA - CRE pair level)

- 具体的に、どの lncRNA - CRE pair が ENCODE-rE2G で予測されているのかを見せるためのデータフレーム
- このデータフレームは、ENCODE-rE2G 側で予測された enhancer 情報を元に `join`している

In [ ]:
# fantabio_HN2_up5p_lncRNA_join_e2g = fantabio_HN2_up5p_in_e2g_lncRNA.select(
#     "transcript_id",
#     "HN2",
#     "gene_id",
#     "gene_symbol",
#     "RNACentral_ID",
#     "annotation_remarks",
#     "cre_id",
#     "cre_name",
#     "screen_ccres",
#     "chip_atlas_antigens",
#     "silencerdbs",
#     "fantom5_cage_peaks",
#     "fantom5_enhancers",
#     "fantabio_cre_class"
# ).join(
#     e2g_join_filter_unique_count,
#     on=["cre_id", "cre_name", "fantabio_cre_class"],
#     how="inner"
# ).sort("HN2", descending=True)


# pl.Config.set_tbl_rows(2)
# display(fantabio_HN2_up5p_lncRNA_join_e2g)

In [ ]:
# # check
# FCHS_354475 = fantabio_HN2_up5p_lncRNA_join_e2g.filter(
#     pl.col("cre_id") == "FCHS_354475"
# )

# display(FCHS_354475)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 8-5: Target Gene Selection

- 8-4 で得た lncRNA - CRE pair（enhancer として他遺伝子を制御）のうち、target 側も upregulated な例を残す
- 出発点は `e2g_join_unique_count` ではなく、8-4 の `fantabio_HN2_up5p_lncRNA_join_e2g`（すでに `isSelfPromoter == False`）
- 照合キーは rE2G の `target_cre_id` と、fantabio の `cre_id`（TSS 側で選んだ代表 CRE）

&nbsp;

&nbsp;

## 8-5-1: Target CRE 上の upregulated transcript

- 8-4 の `target_cre_id` が、HN2 up5p + ATAC diff analysis の `cre_id` に含まれるか確認する
- このセルは target 側の transcript 一覧であり、enhancer 側の絞り込みではない

In [ ]:
# # 上記の enhancer として機能している CRE のうち、標的となっている CRE (target_cre_id) が、HN2 の up5p の diff analysis で upregulated transcripts として含まれているか確認する
# e2g_lncRNA_target = fantabio_HN2_up5p_diff_analysis_atac.filter(
#     pl.col("cre_id").is_in(fantabio_HN2_up5p_lncRNA_join_e2g.get_column("target_cre_id").implode())
# )

# pl.Config.set_tbl_rows(70)
# print(f"e2g_lncRNA_target: {e2g_lncRNA_target.select('gene_symbol', "transcript_id", "gene_id", "HN2").unique().sort('HN2', descending=True)}")

# pl.Config.set_tbl_rows(2)
# display(e2g_lncRNA_target)

&nbsp;

&nbsp;

&nbsp;

## 8-5-2: 8-4 pair に戻して絞る

- 8-4 の lncRNA-enhancer pair を、8-5-1 の target CRE で `inner` join する
- enhancer-derived lncRNA と target 側 transcript の両方を 1 行に残す
- `e2g_join_unique_count` は使わない（self-promoter と、lncRNA 以外の enhancer が混ざるため）


In [ ]:
# # Attach upregulated target transcripts onto the 8-4 lncRNA-enhancer pairs
# e2g_lncRNA_target_filter = fantabio_HN2_up5p_lncRNA_join_e2g.join(
#     e2g_lncRNA_target.select(
#         pl.col("cre_id").alias("target_cre_id"),
#         pl.col("transcript_id").alias("target_transcript_id"),
#         pl.col("gene_symbol").alias("target_gene_symbol"),
#         pl.col("gene_id").alias("target_gene_id"),
#         pl.col("transcript_type").alias("target_transcript_type"),
#         pl.col("HN2").alias("target_HN2"),
#     ),
#     on="target_cre_id",
#     how="inner",
# ).sort("HN2", descending=True)


# pl.Config.set_tbl_rows(2)
# print(f"lncRNA-enhancer pairs with upregulated target CRE: {e2g_lncRNA_target_filter.select(['cre_id', 'target_cre_id']).n_unique()}")
# display(e2g_lncRNA_target_filter)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 8-5: Target Gene Selection (Permissive)

- 8-4 の lncRNA - CRE pair のうち、target 側が upregulated かを、diff analysis 適用前の up5p 全体で見る
- 8-5-2 では target CRE も ATAC diff でシグナルがあるものに限っていた。ここではその制約を外す
- 出発点は同じく 8-4 の `fantabio_HN2_up5p_lncRNA_join_e2g`

&nbsp;

&nbsp;

## 8-5-1 (Permissive): Target CRE 上の upregulated transcript

- 8-4 の `target_cre_id` が、HN2 up5p（diff analysis 前）の `cre_id` に含まれるか確認する
- このセルは target 側の transcript 一覧であり、enhancer 側の絞り込みではない

In [ ]:
# fantabio_HN2_up5p = pl.read_parquet(
#     "../../out/fantabio/parquet_file/fantabio_HN2_up5p_v260721.parquet"
# )


# pl.Config.set_tbl_rows(2)
# display(fantabio_HN2_up5p)

In [ ]:
# # 上記の enhancer として機能している CRE のうち、標的となっている CRE (target_cre_id) が、HN2 の up5p transcript lists (diff analysis 適用前)で upregulated transcripts として含まれているか確認する
# e2g_lncRNA_target_permissive = fantabio_HN2_up5p.filter(
#     pl.col("cre_id").is_in(fantabio_HN2_up5p_lncRNA_join_e2g.get_column("target_cre_id").implode())
# )

# pl.Config.set_tbl_rows(110)
# print(f"e2g_lncRNA_target: {e2g_lncRNA_target_permissive.select('gene_symbol', "transcript_id", "gene_id", "HN2").unique().sort('HN2', descending=True)}")

# pl.Config.set_tbl_rows(2)
# display(e2g_lncRNA_target_permissive)

&nbsp;

&nbsp;

&nbsp;

## 8-5-2 (Permissive): 8-4 pair に戻して絞る

- 8-4 の lncRNA-enhancer pair を、8-5-1 (Permissive) の target CRE で `inner` join する
- enhancer-derived lncRNA と target 側 transcript の両方を 1 行に残す
- 8-5-2 と同じ形だが、target 側は ATAC diff analysis を課さない


In [ ]:
# # Attach permissive upregulated target transcripts onto the 8-4 lncRNA-enhancer pairs
# e2g_lncRNA_target_permissive_filter = fantabio_HN2_up5p_lncRNA_join_e2g.join(
#     e2g_lncRNA_target_permissive.select(
#         pl.col("cre_id").alias("target_cre_id"),
#         pl.col("transcript_id").alias("target_transcript_id"),
#         pl.col("gene_symbol").alias("target_gene_symbol"),
#         pl.col("gene_id").alias("target_gene_id"),
#         pl.col("transcript_type").alias("target_transcript_type"),
#         pl.col("HN2").alias("target_HN2"),
#     ),
#     on="target_cre_id",
#     how="inner",
# ).sort("HN2", descending=True)

# pl.Config.set_tbl_rows(2)
# print(
#     f"lncRNA-enhancer pairs with permissive upregulated target CRE: "
#     f"{e2g_lncRNA_target_permissive_filter.select(['cre_id', 'target_cre_id']).n_unique()}"
# )
# display(e2g_lncRNA_target_permissive_filter)


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 9. Architectural RNA 

- この enhancer derived lncRNA が Architectural RNA の機能を持っていそうか調べる

In [ ]:
# prjdb12380_prjdb9451_heat_result = pl.read_csv(
#     "../../out/integrated_analysis/prjdb12380_prjdb9541/pydeseq2_heat_impr_vs_conv_results.tsv",
#     separator="\t",
#     n_threads=16
# ).filter(
#     (pl.col("log2FoldChange") > 0) &
#     (pl.col("padj") < 0.05)
# )

# display(prjdb12380_prjdb9451_heat_result)

In [ ]:
# prjdb12380_prjdb9451_heat_result_join = prjdb12380_prjdb9451_heat_result.join(
#     fantabio_HN2_up5p_in_e2g_lncRNA,
#     on="gene_id",
#     how="inner"
# )


# display(prjdb12380_prjdb9451_heat_result_join)